In [8]:
import numpy as np
from numpy import ndarray
import torch
from torch import nn



### Реализация BatchNorm.

In [2]:
class BatchNorm1d:
    def __init__(
            self,
            num_features: int,
            eps: float = 1e-5,
            momentum: float = 0.1,
            train: bool = True
    ):
        self.num_features = num_features
        self.eps = eps
        self.m = momentum
        self.training = train

        # Обучаемые параметры γ и β
        self.weight = np.ones((1, self.num_features))   # γ
        self.bias = np.zeros((1, self.num_features))    # β

        # Скользящие статистики для инференса
        self.running_mean = np.zeros((1, self.num_features))
        self.running_var = np.ones((1, self.num_features))

    def forward(self, x: ndarray) -> ndarray:
        if self.training:
            batch_size = x.shape[0]

            # Смещённая оценка среднего и дисперсии по батчу.
            self.batch_mean = np.mean(x, axis=0, keepdims=True)
            self.batch_var = np.var(x, axis=0, keepdims=True)

            # Нормализация по статистикам батча
            self.x_norm = (x - self.batch_mean) / np.sqrt(self.batch_var + self.eps)

            # Несмещённая дисперсия для running_var
            unbiased_var = self.batch_var * batch_size / (batch_size - 1)

            # Обновление скользящих статистик
            self.running_mean = (
                (1 - self.m) * self.running_mean
                + self.m * self.batch_mean
            )
            self.running_var = (
                (1 - self.m) * self.running_var
                + self.m * unbiased_var
            )

        else:
            # В режиме инференса используем накопленные статистики
            self.x_norm = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)

        # Масштабирование и сдвиг
        output = self.weight * self.x_norm + self.bias
        return output

    def backward(self, grad: ndarray) -> ndarray:
        std_inv = self.weight / np.sqrt(self.batch_var + self.eps)        # (1, N)

        # Градиенты обучаемых параметров
        grad_weight = (grad * self.x_norm).sum(axis=0, keepdims=True)
        grad_bias = grad.sum(axis=0, keepdims=True)

        # Градиент по входу
        grad_x = std_inv * (
            grad
            - grad.mean(axis=0, keepdims=True)                                # (1, N)
            - self.x_norm * (grad * self.x_norm).mean(axis=0, keepdims=True)  # (bs, N)
        )

        return grad_x, grad_weight, grad_bias

    def __call__(self, x: ndarray) -> ndarray:
        return self.forward(x)
    
    

In [3]:
# PyTorch
input = torch.normal(16, 100, size=(8, 10), dtype=torch.float32, requires_grad=True)

bn = nn.BatchNorm1d(10)
norm = bn(input)

arr = torch.normal(3, 4, size=norm.shape, dtype=torch.float32)

l = norm * arr
loss = l.sum()

loss.backward()

In [4]:
# Прямой и обратный проход в ручную
input_np_1 = input.detach().numpy()                  # (bs, N)
arr_np_1 = arr.numpy()

bs, N = input_np_1.shape

weight = np.ones((1, N), dtype=np.float32)
bias = np.zeros((1, N), dtype=np.float32)

# Forward
bn_0 = input_np_1.sum(axis=0, keepdims=True)         # (1, N)
bn_mean = bn_0 / bs                                  # (1, N)
bn_1 = input_np_1 - bn_mean                          # (bs, N)
bn_2 = bn_1**2                                       # (bs, N)
bn_3 = bn_2.sum(axis=0, keepdims=True)               # (1, N)
bn_var = bn_3 / bs                                   # (1, N)
bn_sqrt_var = (bn_var + 1e-5)**0.5                   # (1, N)
norm_x = bn_1 / bn_sqrt_var                          # (bs, N)
bn_4 = norm_x * weight                               # (bs, N)
y = bn_4 + bias                                      # (bs, N)

l_np = y * arr_np_1


# Backward
d_l_np = np.ones_like(l_np, dtype=np.float32)        # (bs, N)
d_y = d_l_np * arr_np_1                              # (bs, N)                               
d_bn_4 = d_y                                         # (bs, N)
d_bias = d_y.sum(axis=0, keepdims=True)              # (1, N)
d_weight = np.sum(d_bn_4 * norm_x, axis=0, keepdims=True)                                  # (1, N)
d_norm_x = d_bn_4 * weight                           # (bs, N)
d_bn_1 = d_norm_x / bn_sqrt_var                      # (bs, N)
d_bn_sqrt_var = - np.sum(d_norm_x * bn_1 / (bn_sqrt_var**2), axis=0, keepdims=True)        # (1, N)
d_bn_var = (d_bn_sqrt_var * (bn_var + 1e-4)**1.5) / 2       # (1, N)
d_bn_3 = d_bn_var / bs                               # (1, N)
d_bn_2 = d_bn_3 * np.ones_like(bn_2)                 # (bs, N)
d_bn_1 += 2 * d_bn_2 * bn_1                          # (bs, N)
d_input_np_1 = d_bn_1                                # (bs, N)
d_bn_mean = -d_bn_1.sum(axis=0, keepdims=True)       # (1, N)
d_bn_0 = d_bn_mean / bs                              # (1, N)
d_input_np_1 += d_bn_0 * np.ones_like(bn_2)          # (bs, N)


In [5]:
input_np_2 = input.detach().numpy()
arr_np_2 = arr.numpy()

bn_np = BatchNorm1d(10)
norm_np = bn_np(input_np_2)

grad_inp, grad_w, grad_b = bn_np.backward(np.ones_like(norm_np)*arr_np_2)

In [6]:
def colored_bool(value: bool) -> str:
    """Возвращает строку True/False с цветовым выделением."""
    GREEN = '\033[92m'
    RED   = '\033[91m'
    RESET = '\033[0m'
    return f"{GREEN}True{RESET}" if value else f"{RED}False{RESET}"


result_1 = np.allclose(d_input_np_1, input.grad.detach().numpy())
result_2 = np.allclose(d_weight,     bn.weight.grad.detach().numpy())
result_3 = np.allclose(d_bias,       bn.bias.grad.detach().numpy())

result_4 = np.allclose(grad_inp, input.grad.detach().numpy())
result_5 = np.allclose(grad_w,   bn.weight.grad.detach().numpy())
result_6 = np.allclose(grad_b,   bn.bias.grad.detach().numpy())

print('Градиенты при прямом и обратном проходе вручную:')
print('    grad input ---', colored_bool(result_1))
print('    grad weight --', colored_bool(result_2))
print('    grad bias ----', colored_bool(result_3))

print('\nГрадиенты, рассчитанные с помощью класса BatchNorm1d:')
print('    grad input ---', colored_bool(result_4))
print('    grad weight --', colored_bool(result_5))
print('    grad bias ----', colored_bool(result_6))

Градиенты при прямом и обратном проходе вручную:
    grad input --- False
    grad weight -- True
    grad bias ---- True

Градиенты, рассчитанные с помощью класса BatchNorm1d:
    grad input --- True
    grad weight -- True
    grad bias ---- True


### Реализация Layer Normalization.

In [7]:
class LayerNorm:
    def __init__(self, num_features: int, eps: float = 1e-5):
        self.num_features = num_features
        self.eps = eps

        # Обучаемые параметры
        self.weight = np.ones((1, num_features))   # γ
        self.bias = np.zeros((1, num_features))    # β

    def forward(self, x: ndarray) -> ndarray:
        # Считаю среднее и дисперсию
        self.mean = x.mean(axis=-1, keepdims=True)          # (bs, 1)
        self.var  = x.var(axis=-1, keepdims=True)           # (bs, 1)

        # Нормализация входных данных
        self.x_norm = (x - self.mean) / np.sqrt(self.var + self.eps)   # (bs, N)

        # Масштабирование и сдвиг
        out = self.weight * self.x_norm + self.bias                    # (bs, N)

        return out

    def backward(self, grad: ndarray) -> ndarray:
        std_inv = self.weight / np.sqrt(self.var + self.eps)   # (bs, N)

        # Градиенты обучаемых параметров
        grad_weight = (grad * self.x_norm).sum(axis=0, keepdims=True)
        grad_bias = grad.sum(axis=0, keepdims=True)

        # Градиент по входу
        grad_x = std_inv * (
            grad
            - grad.mean(axis=-1, keepdims=True)
            - self.x_norm * (grad * self.x_norm).mean(axis=-1, keepdims=True)
        )

        return grad_x, grad_weight, grad_bias

    def __call__(self, x: ndarray) -> ndarray:
        return self.forward(x)
    
    